# Лабораторная работа №4. Исследовательский анализ и визуализация данных

**Дисциплина:** Новые технологии в РПС  
**Вариант:** №9 (Текстовые отзывы с сервиса *Otzovik*, классы 1–5 звёзд)  
**Выполнил:** студент Кондратенко А.С. (гр. M_PIN_RIS_2607)  

--- 
## Цель работы
Проведение исследовательского анализа текстовых данных (EDA), расчет описательных статистик, токенизация, фильтрация, морфологический анализ (лемматизация) с использованием библиотек `nltk` и `pymorphy3`, а также визуализация распределений и частотности слов средствами `matplotlib` и `seaborn`.

## 0. Подготовка окружения, загрузка библиотек и данных

In [ ]:
import os
from collections import Counter
from pathlib import Path
import re
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
import pandas as pd
import pymorphy3
import seaborn as sns

# Автозагрузка данных в среде Google Colab
if not Path('dataset').exists():
    print('Клонирование репозитория в Google Colab...')
    os.system('git clone https://github.com/Alexeyphysics/ntrps_lab4.git _tmp_repo')
    if Path('_tmp_repo/dataset').exists():
        os.system('cp -r _tmp_repo/dataset ./dataset')
    os.system('rm -rf _tmp_repo')

# Загрузка стоп-слов русского языка
try:
    stop_words = set(stopwords.words('russian'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    stop_words = set(stopwords.words('russian'))

morph = pymorphy3.MorphAnalyzer()
print('Окружение готово к анализу!')

--- 
## 1. Загрузка данных в DataFrame (Пункт 1)

In [ ]:
records = []
root_path = Path('dataset').resolve()

for class_folder in sorted(root_path.iterdir()):
    if class_folder.is_dir() and class_folder.name.isdigit():
        rating = int(class_folder.name)
        for f in sorted(class_folder.glob('*.txt')):
            lines = f.read_text(encoding='utf-8').strip().split('\n', 1)
            title = lines[0].strip() if lines else 'Без заголовка'
            body = lines[1].strip() if len(lines) > 1 else ''
            records.append((rating, title, body))

df = pd.DataFrame(records, columns=['rating', 'title', 'text'])
print(f'Загружено отзывов: {len(df)}')
df.head()

--- 
## 2. Форматирование названий колонок (Пункт 2)
Приведение имен столбцов к стилю snake_case в нижнем регистре на английском языке.

In [ ]:
df.columns = [re.sub(r'\W+', '_', col.strip().lower()).strip('_') for col in df.columns]
print('Имена колонок:', df.columns.tolist())
df.head(3)

--- 
## 3. Проверка и обработка пропусков (Пункт 3)

In [ ]:
print('Пропуски по столбцам:')
print(df.isna().sum())

# Удаление записей с пустым текстом
df = df.dropna().copy()
df = df[df['text'].str.strip().str.len() > 0].reset_index(drop=True)
print(f'Строк после очистки: {len(df)}')

--- 
## 4. Токенизация и расчет количества слов (Пункт 4)

In [ ]:
# Подсчет количества слов через регулярное выражение
df['word_count'] = df['text'].apply(lambda t: len(re.findall(r'\b\w+\b', str(t))))
df[['rating', 'word_count', 'title']].head(8)

--- 
## 5. Статистический анализ и график распределения Boxplot (Пункт 5)

In [ ]:
print('Описательная статистика числовых столбцов (describe):')
display(df[['rating', 'word_count']].describe())

# Построение графика Boxplot
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='rating', y='word_count', hue='rating', palette='Set2', legend=False)
plt.title('Распределение длины отзывов по оценкам (Boxplot)', fontsize=12)
plt.xlabel('Класс рейтинга (★)', fontsize=11)
plt.ylabel('Количество слов', fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

--- 
## 6. Фильтрация по количеству слов (Пункт 6)
Функция фильтрует DataFrame, оставляя строки с длиной не более max_len слов.

In [ ]:
def filter_by_max_words(data: pd.DataFrame, max_len: int) -> pd.DataFrame:
    return data[data['word_count'] <= max_len].sort_values(by='word_count', ascending=False)

short_df = filter_by_max_words(df, max_len=50)
print(f'Отзывов длиной не более 50 слов: {len(short_df)}')
short_df[['rating', 'word_count', 'title']].head(5)

--- 
## 7. Фильтрация по метке класса (Пункт 7)
Функция фильтрации по конкретной оценке.

In [ ]:
def filter_by_rating(data: pd.DataFrame, target_rating: int) -> pd.DataFrame:
    return data[data['rating'] == target_rating].reset_index(drop=True)

one_star_df = filter_by_rating(df, target_rating=1)
print(f'Отзывов с оценкой ★ 1: {len(one_star_df)}')
one_star_df[['rating', 'word_count', 'title']].head(5)

--- 
## 8. Группировка по классу со статистикой длин (Пункт 8)

In [ ]:
grouped_stats = df.groupby('rating')['word_count'].agg(
    min_words='min',
    max_words='max',
    mean_words='mean',
    median_words='median',
    total_count='count'
).reset_index()

print('Сводная таблица параметров длины текстов по классам:')
grouped_stats

--- 
## 9-10. Очистка текста, лемматизация и частотные гистограммы (Пункты 9 и 10)

In [ ]:
def get_top_lemmatized_words(data: pd.DataFrame, rating_val: int, top_n: int = 15) -> Dict[str, int]:
    subset = data[data['rating'] == rating_val]
    custom_stops = stop_words.union({'это', 'свой', 'который', 'весь', 'мочь', 'год'})
    lemmas = []
    for text in subset['text']:
        words = re.findall(r'[а-яёА-ЯЁ]{3,}', str(text).lower())
        for w in words:
            if w not in custom_stops:
                norm = morph.parse(w)[0].normal_form
                if norm not in custom_stops and len(norm) > 2:
                    lemmas.append(norm)
    return dict(Counter(lemmas).most_common(top_n))

def plot_word_freq(word_freq: Dict[str, int], rating_val: int, bar_color: str) -> None:
    words = list(word_freq.keys())[::-1]
    counts = list(word_freq.values())[::-1]
    plt.figure(figsize=(9, 5))
    bars = plt.barh(words, counts, color=bar_color)
    plt.title(f'Топ-15 лемм в отзывах с оценкой ★ {rating_val}', fontsize=12)
    plt.xlabel('Частота упоминания', fontsize=11)
    plt.ylabel('Лемма (нормальная форма)', fontsize=11)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    for bar in bars:
        plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, str(int(bar.get_width())), va='center')
    plt.show()

# Частотный анализ для негативных отзывов (1 звезда)
top_neg = get_top_lemmatized_words(df, rating_val=1, top_n=15)
plot_word_freq(top_neg, rating_val=1, bar_color='#e74c3c')

# Частотный анализ для позитивных отзывов (5 звезд)
top_pos = get_top_lemmatized_words(df, rating_val=5, top_n=15)
plot_word_freq(top_pos, rating_val=5, bar_color='#2ecc71')